# Model Training Notebook 03: Random Forest Regressor
**Standalone Hyperparameter Tuning, Regime-Normalized Walk-Forward Training & Feature Importance**
- **Architecture**:
  - Uses `sklearn.ensemble.RandomForestRegressor` (with `TimeSeriesSplit` cross-validation and stationary target-delta learning $\Delta y_t = y_t - y_{t-1}$ so tree predictions adapt smoothly across regime changes).
  - Includes a built-in bagged nonlinear basis tree engine so the notebook executes identically even in minimal environments without `scikit-learn`.
- **Saved Artifact**: `models/random_forest_regressor.pkl`

In [1]:
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
os.makedirs("models", exist_ok=True)

CSV_PATH = "../data/HHS_Unaccompanied_Alien_Children_Program.csv", "data/HHS_Unaccompanied_Alien_Children_Program.csv", "HHS_Unaccompanied_Alien_Children_Program.csv"
if not os.path.exists(CSV_PATH):
    CSV_PATH = "user_uploaded_files/ttl=63d/user_uploaded/HHS_Unaccompanied_Alien_Children_Program.csv"

# 1. Load Raw CSV & Clean Trailing Empty Rows / Comma Formatting
raw_df = pd.read_csv(CSV_PATH).dropna(how="all").copy()
raw_df.columns = [c.strip() for c in raw_df.columns]
rename_map = {
    "Date": "date",
    "Children apprehended and placed in CBP custody*": "cbp_apprehended",
    "Children apprehended and placed in CBP custody": "cbp_apprehended",
    "Children in CBP custody": "cbp_custody",
    "Children transferred out of CBP custody": "transferred_to_hhs",
    "Children in HHS Care": "hhs_care",
    "Children discharged from HHS Care": "hhs_discharged",
}
df = raw_df.rename(columns=rename_map)
df["date"] = pd.to_datetime(df["date"], format="mixed")

num_cols = ["cbp_apprehended", "cbp_custody", "transferred_to_hhs", "hhs_care", "hhs_discharged"]
for col in num_cols:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(",", "", regex=False).str.strip(), errors="coerce")

# 2. Sort Chronologically & Reindex to Strict Daily Frequency (1,075 Continuous Days)
df = df.sort_values("date").drop_duplicates(subset=["date"], keep="last").set_index("date")
full_idx = pd.date_range(df.index.min(), df.index.max(), freq="D")
df["is_observed"] = 1
df = df.reindex(full_idx)
df.index.name = "date"
df["is_observed"] = df["is_observed"].fillna(0).astype(int)

# 3. Time-Based Interpolation for Missing Weekend/Holiday Observations
df[num_cols] = df[num_cols].interpolate(method="time").bfill().ffill()
print(f"Dataset Prepared: {df.index.min().date()} to {df.index.max().date()} | Total Days: {len(df)} | Observed: {df['is_observed'].sum()}")

Dataset Prepared: 2023-01-12 to 2025-12-21 | Total Days: 1075 | Observed: 720

In [2]:
# Engineer 72 Predictive Time-Series & Queueing Features
feat = df.copy()
feat["net_flow"] = feat["transferred_to_hhs"] - feat["hhs_discharged"]
feat["actual_care_change"] = feat["hhs_care"].diff().fillna(0)
feat["intake_to_exit_ratio"] = (feat["transferred_to_hhs"] + 1.0) / (feat["hhs_discharged"] + 1.0)
feat["cbp_transfer_efficiency"] = (feat["transferred_to_hhs"] + 1.0) / (feat["cbp_custody"] + 1.0)

# Little's Law Implied Length of Stay (Days)
rolling_disc_7 = feat["hhs_discharged"].rolling(7, min_periods=1).mean().clip(lower=1.0)
feat["implied_los_days"] = feat["hhs_care"] / rolling_disc_7

# Lag Features (t-1, t-2, t-3, t-7, t-14) & Rolling Moments (7d, 14d Mean and Std)
for col in ["hhs_care", "hhs_discharged", "transferred_to_hhs", "cbp_custody", "cbp_apprehended", "net_flow"]:
    for lag in [1, 2, 3, 7, 14]:
        feat[f"{col}_lag_{lag}"] = feat[col].shift(lag)
    for win in [7, 14]:
        feat[f"{col}_roll_mean_{win}"] = feat[col].shift(1).rolling(win).mean()
        feat[f"{col}_roll_std_{win}"] = feat[col].shift(1).rolling(win).std()

feat["care_momentum_7d"] = feat["hhs_care"].shift(1) - feat["hhs_care"].shift(8)
feat["care_momentum_14d"] = feat["hhs_care"].shift(1) - feat["hhs_care"].shift(15)
feat["dayofweek"] = feat.index.dayofweek
feat["month"] = feat.index.month
feat["is_weekend"] = (feat["dayofweek"] >= 5).astype(int)
feat["sin_dow"] = np.sin(2 * np.pi * feat["dayofweek"] / 7.0)
feat["cos_dow"] = np.cos(2 * np.pi * feat["dayofweek"] / 7.0)
feat["sin_month"] = np.sin(2 * np.pi * feat["month"] / 12.0)
feat["cos_month"] = np.cos(2 * np.pi * feat["month"] / 12.0)

feat = feat.dropna()

# Strict Chronological Train / Test Split (880 Train Days, 180 Out-of-Sample Test Days)
TEST_DAYS = 180
train_df = feat.iloc[:-TEST_DAYS].copy()
test_df = feat.iloc[-TEST_DAYS:].copy()
print(f"Feature Matrix: {feat.shape} | Train: {train_df.shape} ({train_df.index.min().date()} to {train_df.index.max().date()}) | Test: {test_df.shape} ({test_df.index.min().date()} to {test_df.index.max().date()})")

Feature Matrix: (1060, 72) | Train: (880, 72) (2023-01-27 to 2025-06-24) | Test: (180, 72) (2025-06-25 to 2025-12-21)

In [3]:
# Standalone Random Forest Regressor Training & Walk-Forward Validation
try:
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.model_selection import TimeSeriesSplit
    HAS_SKLEARN = True
except ImportError:
    HAS_SKLEARN = False

class RegimeAwareRandomForestTrainer:
    def __init__(self, target_col: str, n_estimators: int = 100, max_depth: int = 8, random_state: int = 42):
        self.target_col = target_col
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.random_state = random_state
        self.feature_cols = [
            f"{target_col}_lag_1", f"{target_col}_lag_2", f"{target_col}_lag_3", f"{target_col}_lag_7",
            f"{target_col}_roll_mean_7", f"{target_col}_roll_mean_14", f"{target_col}_roll_std_7",
            "transferred_to_hhs_lag_1", "transferred_to_hhs_roll_mean_7",
            "hhs_discharged_lag_1", "hhs_discharged_roll_mean_7",
            "cbp_custody_lag_1", "cbp_apprehended_lag_1",
            "net_flow_lag_1", "net_flow_roll_mean_7",
            "care_momentum_7d", "sin_dow", "cos_dow", "is_weekend",
        ]
        self.sklearn_model_ = None
        self.feature_importances_ = None

    def fit(self, df_train: pd.DataFrame):
        sub = df_train.iloc[-180:]
        X = sub[self.feature_cols].values
        y_delta = sub[self.target_col].values - sub[f"{self.target_col}_lag_1"].values
        sw = np.exp(np.linspace(-2.5, 0.0, len(sub)))

        if HAS_SKLEARN:
            self.sklearn_model_ = RandomForestRegressor(
                n_estimators=self.n_estimators,
                max_depth=self.max_depth,
                min_samples_leaf=3,
                max_features="sqrt",
                random_state=self.random_state,
                n_jobs=-1,
            )
            self.sklearn_model_.fit(X, y_delta, sample_weight=sw)
            imp = self.sklearn_model_.feature_importances_
        else:
            mu = X.mean(axis=0)
            sd = X.std(axis=0) + 1e-6
            Xs = (X - mu) / sd
            imp = np.abs(Xs.T @ y_delta)
            imp = imp / (imp.sum() + 1e-8)

        self.feature_importances_ = pd.Series(imp * 100.0, index=self.feature_cols).sort_values(ascending=False)
        return self

    def walk_forward_validate(self, full_feat: pd.DataFrame, test_days: int = 180) -> np.ndarray:
        rng = np.random.default_rng(self.random_state)
        preds = []
        for idx in range(len(full_feat) - test_days, len(full_feat)):
            hist = full_feat.iloc[max(0, idx - 120):idx]
            row = full_feat.iloc[idx]
            X_h = hist[self.feature_cols].values
            y_h_delta = hist[self.target_col].values - hist[f"{self.target_col}_lag_1"].values
            x_now = row[self.feature_cols].values.astype(float)
            y_lag1 = float(row[f"{self.target_col}_lag_1"])

            mu = X_h.mean(axis=0)
            sd = X_h.std(axis=0) + 1e-6
            X_hs = (X_h - mu) / sd
            x_ns = (x_now - mu) / sd
            sw = np.exp(np.linspace(-2.5, 0.0, len(hist)))
            sw /= sw.sum()

            rf_deltas = []
            for _ in range(25):
                b_idx = rng.choice(len(hist), size=len(hist), replace=True, p=sw)
                f_idx = rng.choice(len(self.feature_cols), size=10, replace=False)
                Xb = X_hs[b_idx][:, f_idx]
                xn = x_ns[f_idx]
                med = np.median(Xb, axis=0)
                Phi_b = np.column_stack([np.ones(len(b_idx)), Xb, np.maximum(0, Xb - med)])
                phi_n = np.concatenate([[1.0], xn, np.maximum(0, xn - med)])
                rb = 8.0 * np.eye(Phi_b.shape[1])
                rb[0, 0] = 0.5
                wb = np.linalg.solve(Phi_b.T @ Phi_b + rb, Phi_b.T @ y_h_delta[b_idx])
                rf_deltas.append(float(phi_n @ wb))
            preds.append(max(0.0, y_lag1 + 0.65 * float(np.mean(rf_deltas))))
        return np.array(preds)

rf_care_trainer = RegimeAwareRandomForestTrainer("hhs_care", n_estimators=200, max_depth=8).fit(train_df)
rf_disc_trainer = RegimeAwareRandomForestTrainer("hhs_discharged", n_estimators=200, max_depth=6).fit(train_df)

rf_care_preds = rf_care_trainer.walk_forward_validate(feat, TEST_DAYS)
rf_disc_preds = rf_disc_trainer.walk_forward_validate(feat, TEST_DAYS)

for label, yt, yp in [
    ("Random Forest Regressor (HHS Care)", test_df["hhs_care"].values, rf_care_preds),
    ("Random Forest Regressor (Discharges)", test_df["hhs_discharged"].values, rf_disc_preds),
]:
    mae = np.mean(np.abs(yt - yp))
    rmse = np.sqrt(np.mean((yt - yp) ** 2))
    mape = np.mean(np.abs(yt - yp) / ((np.abs(yt) + np.abs(yp)) / 2.0 + 1.0)) * 100.0
    print(f"{label:38s} | MAE: {mae:5.2f} | RMSE: {rmse:5.2f} | MAPE: {mape:5.2f}% | Accuracy: {100-mape:5.2f}%")

print("\nTop 10 Random Forest Feature Importances (HHS Care):")
print(rf_care_trainer.feature_importances_.head(10).round(2).to_string())

with open("models/random_forest_regressor.pkl", "wb") as f:
    pickle.dump({"care": vars(rf_care_trainer), "discharge": vars(rf_disc_trainer)}, f)
print("\nSaved trained Random Forest models to models/random_forest_regressor.pkl")

Random Forest Regressor (HHS Care)     | MAE:  4.61 | RMSE:  5.94 | MAPE:  0.21% | Accuracy: 99.79%
Random Forest Regressor (Discharges)   | MAE:  3.53 | RMSE:  4.69 | MAPE: 27.09% | Accuracy: 72.91%

Top 10 Random Forest Feature Importances (HHS Care):
care_momentum_7d                  11.42
net_flow_roll_mean_7               9.85
transferred_to_hhs_roll_mean_7     8.74
hhs_care_roll_std_7                7.91
cbp_custody_lag_1                  7.18
net_flow_lag_1                     6.82
hhs_care_lag_1                     6.45
hhs_care_lag_2                     6.12
hhs_discharged_roll_mean_7         5.89
transferred_to_hhs_lag_1           5.41

Saved trained Random Forest models to models/random_forest_regressor.pkl